In [ ]:
# Install once in Colab / Kaggle
!pip install -q datasets transformers tqdm

In [ ]:
from datasets import load_dataset

In [ ]:
DATASET_NAME = "gretelai/synthetic_text_to_sql"

TARGET_TOKENS = 50_000_000

MIN_CHARS = 80
MAX_CHARS = 100_000

OUTPUT_FILE = "bash_corpus.txt"

In [ ]:
dataset = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

In [ ]:
# def is_good_shell(code, min_code_ratio=0.35):
#     lines = [line.strip() for line in code.splitlines() if line.strip()]

#     if len(lines) < 3:
#         return False

#     code_lines = 0
#     comment_lines = 0

#     for line in lines:
#         if line.startswith("#") and not line.startswith("#!"):
#             comment_lines += 1
#         else:
#             code_lines += 1

#     code_ratio = code_lines / len(lines)

#     return code_ratio >= min_code_ratio

In [ ]:
import pandas as pd

rows = []

for example in dataset:
    rows.append({
        "sql_prompt": example["sql_prompt"],
        "sql": example["sql"],
        "sql_context": example["sql_context"]
    })

df = pd.DataFrame(rows)

print(df.head())
print(f"Total rows: {len(df):,}")

                                          sql_prompt  \
0  What is the total volume of timber sold by eac...   
1  List all the unique equipment types and their ...   
2  How many marine species are found in the South...   
3  What is the total trade value and average pric...   
4  Find the energy efficiency upgrades with the h...   

                                                 sql  \
0  SELECT salesperson_id, name, SUM(volume) as to...   
1  SELECT equipment_type, SUM(maintenance_frequen...   
2  SELECT COUNT(*) FROM marine_species WHERE loca...   
3  SELECT trader_id, stock, SUM(price * quantity)...   
4  SELECT type, cost FROM (SELECT type, cost, ROW...   

                                         sql_context  
0  CREATE TABLE salesperson (salesperson_id INT, ...  
1  CREATE TABLE equipment_maintenance (equipment_...  
2  CREATE TABLE marine_species (name VARCHAR(50),...  
3  CREATE TABLE trade_history (id INT, trader_id ...  
4  CREATE TABLE upgrades (id INT, cost FLOAT, typ..

In [ ]:
all_text = "\n".join(
    df["sql_prompt"].fillna("").astype(str)
    + "\n"
    + df["sql_context"].fillna("").astype(str)
    + "\n"
    + df["sql"].fillna("").astype(str)
)

print(f"Total characters: {len(all_text):,}")

Total characters: 48,957,963


In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
import ast
import json
import re
import os


DATASET_NAME_TWO = "cwolff/text-to-sql-wip-100k"
OUTPUT_FILE = "sql_wip_clean_100mb.txt"

In [ ]:
dataset_two = load_dataset(
    DATASET_NAME_TWO,
    split="train",
    streaming=True
)

In [ ]:
import pandas as pd

rows = []

TARGET_BYTES = 115 * 1024 * 1024  # 1 MB
current_bytes = 0

for example in dataset_two:

    row = {
        "sql_prompt": example["question"],
        "sql": example["sql statament"],
        "sql_context": example["Full schema"]
    }

    # Approximate actual UTF-8 size of this row
    row_text = (
        str(row["sql_prompt"])
        + str(row["sql"])
        + str(row["sql_context"])
    )

    row_bytes = len(row_text.encode("utf-8"))

    # Stop before exceeding ~1 MB
    if current_bytes + row_bytes > TARGET_BYTES:
        break

    rows.append(row)
    current_bytes += row_bytes


df = pd.DataFrame(rows)

print(df.head())
print(f"Total rows: {len(df):,}")
print(f"Approx data size: {current_bytes / (1024**2):.3f} MB")

                                          sql_prompt  \
0  What is the total number of comment threads th...   
1  What is the average priority of all active tas...   
2  Hey, can you get me the last modified dates fo...   
3  1. Identify the user's email address. 2. Acces...   
4  Whar is the phonenumber of the office wer the ...   

                                                 sql  \
0  SELECT SUM(CASE WHEN responseCount > 5 THEN 1 ...   
1  SELECT AVG(priority_) FROM ACT_RU_TASK WHERE i...   
2  SELECT 'egeis_assignment' AS table_name, MAX(l...   
3  SELECT u.email  FROM users u  JOIN access_card...   
4  SELECT e.last_name, d.dept_name, l.phone_numbe...   

                                         sql_context  
0  CREATE TABLE User (   id Integer NOT NULL UNIQ...  
1  CREATE TABLE ACT_APP_DEPLOYMENT_RESOURCE (   I...  
2  CREATE TABLE egeis_employeetype (   id BigInt ...  
3  CREATE TABLE master (   type Varchar NOT NULL,...  
4  CREATE TABLE employee (   employee_id TEXT NOT..

In [ ]:
all_text_two = "\n".join(
    df["sql_prompt"].fillna("").astype(str)
    + "\n"
    + df["sql_context"].fillna("").astype(str)
    + "\n"
    + df["sql"].fillna("").astype(str)
)

print(f"Total characters: {len(all_text_two):,}")

Total characters: 120,561,858


In [ ]:
total_text = all_text + "\n\n### End\n\n" + all_text_two

In [ ]:
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

no_of_chunks = 4
chunk_size = len(total_text) // no_of_chunks

total_tokens = 0

for i in range(no_of_chunks):
    start = i * chunk_size
    end = None if i == no_of_chunks - 1 else (i + 1) * chunk_size

    total_tokens += len(tokenizer.encode(total_text[start:end]))

print(f"Total tokens: {total_tokens:,}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (12763094 > 1024). Running this sequence through the model will result in indexing errors


Total tokens: 57,812,623


In [ ]:
import numpy as np
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

chunk_size = 1_000_000  # characters
output = "sql_tokens.bin"

with open(output, "wb") as f:
    for i in range(0, len(total_text), chunk_size):
        chunk = total_text[i:i+chunk_size]
        ids = tokenizer.encode(chunk, add_special_tokens=False)
        np.array(ids, dtype=np.uint16).tofile(f)

        print(f"{i / len(total_text) * 100:.1f}%")

print("DONE")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299531 > 1024). Running this sequence through the model will result in indexing errors


0.0%
0.6%
1.2%
1.8%
2.4%
2.9%
3.5%
4.1%
4.7%
5.3%
5.9%
6.5%
7.1%
7.7%
8.3%
8.8%
9.4%
10.0%
10.6%
11.2%
11.8%
12.4%
13.0%
13.6%
14.2%
14.7%
15.3%
15.9%
16.5%
17.1%
17.7%
18.3%
18.9%
19.5%
20.1%
20.6%
21.2%
21.8%
22.4%
23.0%
23.6%
24.2%
24.8%
25.4%
26.0%
26.5%
27.1%
27.7%
28.3%
28.9%
29.5%
30.1%
30.7%
31.3%
31.9%
32.4%
33.0%
33.6%
34.2%
34.8%
35.4%
36.0%
36.6%
37.2%
37.8%
38.3%
38.9%
39.5%
40.1%
40.7%
41.3%
41.9%
42.5%
43.1%
43.7%
44.2%
44.8%
45.4%
46.0%
46.6%
47.2%
47.8%
48.4%
49.0%
49.6%
50.1%
50.7%
51.3%
51.9%
52.5%
53.1%
53.7%
54.3%
54.9%
55.5%
56.0%
56.6%
57.2%
57.8%
58.4%
59.0%
59.6%
60.2%
60.8%
61.3%
61.9%
62.5%
63.1%
63.7%
64.3%
64.9%
65.5%
66.1%
66.7%
67.2%
67.8%
68.4%
69.0%
69.6%
70.2%
70.8%
71.4%
72.0%
72.6%
73.1%
73.7%
74.3%
74.9%
75.5%
76.1%
76.7%
77.3%
77.9%
78.5%
79.0%
79.6%
80.2%
80.8%
81.4%
82.0%
82.6%
83.2%
83.8%
84.4%
84.9%
85.5%
86.1%
86.7%
87.3%
87.9%
88.5%
89.1%
89.7%
90.3%
90.8%
91.4%
92.0%
92.6%
93.2%
93.8%
94.4%
95.0%
95.6%
96.2%
96.7%
97.3%
97.9%
98.5%
99.1%
99.